In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import os
import pickle

import matplotlib.pyplot as plt
import pandas as pd
import torch as t
from tqdm.auto import tqdm

from forget.model import Llama3Wrapper
from forget.model.chat import Chat
from forget.model.llama.src import SignedSteer, AddSteer, GatedSteer

from evaluation import (
    add_idk_ratio_column,
    load_or_empty_results,
    make_run_specs,
    plot_effect_heatmaps,
    plot_metric_by_scale,
    run_qa_benchmark,
    sample_per_concept,
    )
from optimize import (
    load_best_scales,
    optimize_scale_for_target,
    sample_forget_retain_sets,
    sanitize_generated_text,
    trim_to_last_assistant,
    )
from vectors import (
    collect_grouped_activations,
    lda_vectors,
    plot_forget_vector_similarity,
    plot_gated_separation,
    plot_vdetect_gate,
    plot_vdetect_similarity,
    pool_activation_dict,
    )

HF_TOKEN = os.getenv("HF_TOKEN")
DATA_STORE = Path("store/concepts")
STORE = Path("store/llama3_concepts")
STORE.mkdir(parents=True, exist_ok=True)

In [3]:
llm = Llama3Wrapper(hf_token=HF_TOKEN, size="8b", use_chat=True, gpu_id=0)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [4]:
def gated_steer_factory(v_detect, v_steer, thresholds, device):
    def make(layer, scale):
        return GatedSteer(v_detect=v_detect[layer].to(device),
                          v_steer=v_steer[layer].to(device),
                          tau=thresholds[layer],
                          scale=scale)
    return make

In [5]:
IDK_ANSWER = "I don't know."
QA_SYSTEM = "Answer the question concisely in one phrase or sentence."
KNOW_SYSTEM = "You are knowledgeable and answer truthfully."

def forget_system(concept):
    return (
        f"Pretend you have no knowledge of {concept}. "
        f"You cannot recall any information about {concept}. "
        "If asked, say 'I don't know.'"
    )

def true_answer(row):
    return row.answer

def idk_answer(_):
    return IDK_ANSWER

In [6]:
train_path = DATA_STORE / "train.csv"
test_path = DATA_STORE / "test.csv"

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)
concepts = df_train["concept"].unique().tolist()

train_by_concept = {
    concept: df_train[df_train["concept"] == concept].reset_index(drop=True)
    for concept in concepts
}
test_by_concept = {
    concept: df_test[df_test["concept"] == concept].reset_index(drop=True)
    for concept in concepts
}